<div style="width:100%; background-color:#181818; color:#f1f1f1; padding:30px 0; text-align:center; border-radius:10px;">
  <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExODU4Zjk2NzZucnlsNDNjaWYzc2NwdXcxeG5tZThhOHV4MzZmYW5qOCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/eaBro4mE3Oe2I/giphy.gif" alt="Reinforced concrete design animation" width="300" style="max-width:100%; border-radius:10px;">
  <h3 style="color:#ffffff; margin:0 0 8px 0;"><b>Slabs, Direct Design Method</b></h3>
  <p style="margin:4px 0;"><b>Authors:</b> Msc. Ing. Carlos Andr&eacute;s Celi S&aacute;nchez &middot; Nicol&aacute;s Mora Bowen &middot; Juan Sebasti&aacute;n Baquero</p>
  <p style="margin:4px 0;"><b>Course:</b> Reinforced Concrete Design II</p>
</div>

### Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from repo__CORE_concreteII_desing import *

### Data

In [2]:
fc = 280
fy = 4200
lx = 500
ly = 500
bw = 100
dvarF = 1.2
Ecb = 200000
Ecs = 200000

WD = 0.030                                      # kg/cm2
WL = 0.020                                      # kg/cm2

qu = 1.2 * WD + 1.6 * WL                        # kg/cm2


panel = 1                                       # 1 interior, 2 border or corner

end_span_cond = 5

condition1 = True                               # There must be a minimum of three continuous spans in each direction
condition2 = max(lx, ly) / min(lx, ly) <= 2     # Panels must be rectangular, with the ratio of the longer to the shorter span (center-to-center of supports) not exceeding 2.
condition3 = True                               # Successive span lengths, center-to-center of supports, in each direction must not differ by more than one-third of the longer span.
condition4 = True                               # Columns may be offset from either axis between centerlines of successive columns by no more than 10 percent of the span (in the direction of the offset).
condition5 = True                               # All loads must be due to gravity only, uniformly distributed over an entire panel, and the unfactored live load must not exceed twice the unfactored dead load.
condition6 = True                               # For a panel with beams between supports on all sides, the relative stiffness of the beams in the two perpendicular directions must satisfy

bv = 30
bh = 20


fi = 0.9
fiC = 0.75
rec= 2.5
tc = 40
tl = 40
alfas = 40
diF = 1.2

print(qu)

0.068


In [3]:
# 8.3.1.1. Two way slabs without interior beams
H = np.ceil(np.max([lx, ly]) / 33 / 5) * 5

#### Condition 6

In [4]:
ala_beam = min(bh - H, 4*H)

match panel:
    case 1:
        #------------ Iner Beam ------------#
        ala_b = bv + 2*ala_beam
        ala_h = H
        area_ala = ala_b * ala_h
        cg_ala = H/2
        I_ala = ala_b*ala_h**(3)/12

        alma_b = bv
        alma_h = bh - H
        area_alma = alma_b * alma_h
        cg_alma = H + alma_h/2
        I_alma = alma_b*alma_h**(3)/12

        areas = np.array([area_ala, area_alma])
        iners = np.array([I_ala, I_alma])
        yi    = np.array([cg_ala, cg_alma])
        a_y   = areas * yi


        ycg = np.sum(a_y) / np.sum(areas)
        
        Ibi = []
        for i in np.arange(0,len(areas)):
            Ibi.append(iners[i] + areas[i]*(yi[i] - ycg)**(2))

        Ib = np.sum(Ibi)

        #------------ Iner Slab ------------#
        L_colb1 = 2 * ly / 2                    # beams spanning in x: half panel on each side, measured in y
        L_colb2 = 2 * lx / 2                    # beams spanning in y: half panel on each side, measured in x
        Is1 = L_colb1 * H**(3) /12
        Is2 = L_colb2 * H**(3) /12

        #------------ αf ------------#     
        alfa_f1 = (Ecb*Ib) / (Ecs*Is1)
        alfa_f2 = (Ecb*Ib) / (Ecs*Is2)

        #------------ Condition 6 (8.10.2.7) ------------#
        ratio6 = alfa_f1*ly**(2) / (alfa_f2*lx**(2))
        condition6 = 0.2 <= ratio6 <= 5.0

        #------------ H min, Table 8.3.1.2 (mks-metric, ACI 318 Appendix C) ------------#
        alfa_fm = (2*alfa_f1 + 2*alfa_f2) / 4   # interior panel: four T-beams
        ln = np.max([lx, ly]) - bv              # clear span in the long direction, face-to-face of beams
        beta = (np.max([lx, ly]) - bv) / (np.min([lx, ly]) - bv)

        if alfa_fm <= 0.2:
            Hmin = np.max([lx, ly]) / 33        # 8.3.1.1 applies
        elif alfa_fm <= 2.0:
            Hmin = max(ln*(0.8 + fy/14000) / (36 + 5*beta*(alfa_fm - 0.2)), 12.5)
        else:
            Hmin = max(ln*(0.8 + fy/14000) / (36 + 9*beta), 9)

        conditions = [condition1, condition2, condition3, condition4, condition5, condition6]
        wy = (lx/ly)**(4)
        porc = wy / (np.ceil(wy) + 1) * 100

        if porc > 75:
            mensaje = 'one way slab'
        elif all(conditions):
            mensaje = 'two way slab - Direct Design Method applies'
            
            print('='*120)
            print(f'Ib = {Ib:,.2f} cm4 (T-beam)')
            print(f'ycg = {ycg:.2f} cm, measured from the top of slab')
            print(f'Is1 = {Is1:,.2f} cm4, Is2 = {Is2:,.2f} cm4')
            print(f'αf1 = {alfa_f1:.2f}, αf2 = {alfa_f2:.2f}, αfm = {alfa_fm:.2f}')
            print(f'Condition 6: 0.2 <= {ratio6:.2f} <= 5.0 -> {condition6}')
            print(f'ln = {ln:.2f} cm, β = {beta:.2f}')
            print(f'H min (8.3.1.2) = {Hmin:.2f} cm, H used = {H:.2f} cm -> {"OK" if H >= Hmin else "INCREASE H"}')
            print('='*120)
            
        else:
            failed = [i + 1 for i, c in enumerate(conditions) if not c]
            mensaje = f'two way slab - all DDM conditions must be satisfied (fails in condition: {failed})'

        print('='*120)
        print(f'Slab behavior = {mensaje}')
        print(f'H min = {H:.2f} cm')
        print(f'%  = {porc:.2f}')
        print('='*120)


        
    case 2:
        pass
    case _:
        raise ValueError("none")

Ib = 20,000.00 cm4 (T-beam)
ycg = 10.00 cm, measured from the top of slab
Is1 = 333,333.33 cm4, Is2 = 333,333.33 cm4
αf1 = 0.06, αf2 = 0.06, αfm = 0.06
Condition 6: 0.2 <= 1.00 <= 5.0 -> True
ln = 470.00 cm, β = 1.00
H min (8.3.1.2) = 15.15 cm, H used = 20.00 cm -> OK
Slab behavior = two way slab - Direct Design Method applies
H min = 20.00 cm
%  = 50.00


### One-Way and Two-Way Action

In [5]:
wy = (lx/ly)**(4)
porc = wy / (np.ceil(wy) + 1) * 100

if porc > 75:
    mensaje = 'one way slab'
elif all(conditions):
    mensaje = 'two way slab - Direct Design Method applies'
else:
    failed = [i + 1 for i, c in enumerate(conditions) if not c]
    mensaje = f'two way slab - all DDM conditions must be satisfied (fails in condition: {failed})'

print('='*120)
print(f'Slab behavior = {mensaje}')
print(f'H min = {H:.2f} cm')
print(f'%  = {porc:.2f}')
print('='*120)

Slab behavior = two way slab - Direct Design Method applies
H min = 20.00 cm
%  = 50.00


### Static Moment

In [6]:
Mo = qu * lx * ly**(2) / 8
print('='*120)
print(f'Static Moment = {Mo / 1e5:.2f} T-m')
print('='*120)

Static Moment = 10.62 T-m


### Distribution

In [7]:
match end_span_cond:
    case 1:
        #------------ Exterior edge unrestrained ------------#
        end_i = 0
        mid = 0.63*Mo
        end_j = 0.75*Mo
        comP_Mo = mid + (end_i + end_j)/2
        if comP_Mo <= 1.05*Mo:
            print('='*120)
            print(f'Mo_comprobation = {comP_Mo / 1e5:.2f} T-m ----> Mo = {Mo / 1e5:.2f} T-m')
            print('='*120)
        else:
            raise ValueError("none")  
    case 2:
        #------------ Slab with beams between all supports ------------#
        end_i = 0.16*Mo
        mid = 0.57*Mo
        end_j = 0.70*Mo
        comP_Mo = mid + (end_i + end_j)/2
        if comP_Mo <= 1.05*Mo:
            print('='*120)
            print(f'Mo_comprobation = {comP_Mo / 1e5:.2f} T-m ----> Mo = {Mo / 1e5:.2f} T-m')
            print('='*120)
        else:
            raise ValueError("none")
    case 3:
        #------------ Slab without beams between interior supports, without an edge beam ------------#
        end_i = 0.26*Mo
        mid = 0.52*Mo
        end_j = 0.70*Mo
        comP_Mo = mid + (end_i + end_j)/2
        if comP_Mo <= 1.05*Mo:
            print('='*120)
            print(f'Mo_comprobation = {comP_Mo / 1e5:.2f} T-m ----> Mo = {Mo / 1e5:.2f} T-m')
            print('='*120)
        else:
            raise ValueError("none")
    case 4:
        #------------ Slab without beams between interior supports, with an edge beam ------------#
        end_i = 0.30*Mo
        mid = 0.50*Mo
        end_j = 0.70*Mo
        comP_Mo = mid + (end_i + end_j)/2
        if comP_Mo <= 1.05*Mo:
            print('='*120)
            print(f'Mo_comprobation = {comP_Mo / 1e5:.2f} T-m ----> Mo = {Mo / 1e5:.2f} T-m')
            print('='*120)
        else:
            raise ValueError("none")
    case 5:
        #------------ Exterior edge fully restrained ------------#
        end_i = 0.65*Mo
        mid = 0.35*Mo
        end_j = 0.65*Mo
        comP_Mo = mid + (end_i + end_j)/2
        if comP_Mo <= 1.05*Mo:
            print('='*120)
            print(f'Mo_comprobation = {comP_Mo / 1e5:.2f} T-m ----> Mo = {Mo / 1e5:.2f} T-m')
            print('='*120)
        else:
            raise ValueError("none")
    case _:
        raise ValueError("none")

Mo_comprobation = 10.62 T-m ----> Mo = 10.62 T-m


#### Lateral Distribution: Column Strips

In [8]:
strip_col = np.min([0.25*lx, 0.25*ly])
strip_midSpan = np.min([lx,ly]) - strip_col

#------ Interior negative moment (ACI 318-14 Table 8.10.5.1) -----#
# CS_i_neg = 75 + 30 * (alfa_fm * np.max([lx,ly]) / np.min([lx,ly]))*(1 - np.max([lx,ly]) / np.min([lx,ly]))
CS_i_neg = 0.75

#------ Positive moment (ACI 318-14 Table 8.10.5.5) -----#
# CS_pos = 60 + 30 * (alfa_fm * np.max([lx,ly]) / np.min([lx,ly]))*(1.5 - np.max([lx,ly]) / np.min([lx,ly]))
CS_pos = 0.75

M_i_neg = [(CS_i_neg * end_i) / 1e5]
M_pos = [(CS_pos * mid) / 1e5]
M_j_neg = [(CS_i_neg * end_j) / 1e5]

M_i_neg_df = pd.DataFrame(M_i_neg, columns= [f'Moment node_i [T-m] (-) / bw = {strip_col:.2f} [cm]'])
M_pos_df = pd.DataFrame(M_pos, columns= [f'Moment mid [T-m] (+)/ bw = {strip_col:.2f} [cm]'])
M_j_neg_df = pd.DataFrame(M_j_neg, columns= [f'Moment node_j [T-m] (-)/ bw = {strip_col:.2f} [cm]'])

Resul = pd.concat([M_i_neg_df, M_pos_df, M_j_neg_df], axis = 1, ignore_index= False)
print('='*120)
print('Lateral Distribution: Column Strips ')
print(f'bw = {strip_col:.2f} [cm]')
print(f'Mo = {Mo / 1e5:.2f} T-m')
print('='* 120)
Resul

Lateral Distribution: Column Strips 
bw = 125.00 [cm]
Mo = 10.62 T-m


,Moment node_i [T-m] (-) / bw = 125.00 [cm],Moment mid [T-m] (+)/ bw = 125.00 [cm],Moment node_j [T-m] (-)/ bw = 125.00 [cm]
0,5.179688,2.789062,5.179688


#### Lateral Distribution: Middle Strips

In [9]:
strip_midSpan = np.min([lx,ly]) - strip_col

#------ Interior negative moment (ACI 318-14 Table 8.10.5.1) -----#
CS_i_neg_mid = 1 - CS_i_neg

#------ Positive moment (ACI 318-14 Table 8.10.5.5) -----#
CS_pos_mid = 1 - CS_pos

M_i_neg_mid = [(CS_i_neg_mid * end_i) / 1e5]
M_pos_mid = [(CS_pos_mid * mid) / 1e5]
M_j_neg_mid = [(CS_i_neg_mid * end_j) / 1e5]

M_i_neg_mid_df = pd.DataFrame(M_i_neg_mid, columns= [f'Moment node_i [T-m] (-) / bw = {strip_midSpan:.2f} [cm]'])
M_pos_mid_df = pd.DataFrame(M_pos_mid, columns= [f'Moment mid [T-m] (+)/ bw = {strip_midSpan:.2f} [cm]'])
M_j_neg_mid_df = pd.DataFrame(M_j_neg_mid, columns= [f'Moment node_j [T-m] (-)/ bw = {strip_midSpan:.2f} [cm]'])

Resul_mid = pd.concat([M_i_neg_mid_df, M_pos_mid_df, M_j_neg_mid_df], axis = 1, ignore_index= False)
print('='*120)
print('Lateral Distribution: Middle Strips ')
print(f'bw = {strip_midSpan:.2f} [cm]')
print(f'Mo = {Mo / 1e5:.2f} T-m')
print('='* 120)
Resul_mid

Lateral Distribution: Middle Strips 
bw = 375.00 [cm]
Mo = 10.62 T-m


,Moment node_i [T-m] (-) / bw = 375.00 [cm],Moment mid [T-m] (+)/ bw = 375.00 [cm],Moment node_j [T-m] (-)/ bw = 375.00 [cm]
0,1.726562,0.929688,1.726562


### Verifaction of Shear Demand vs Capacity

In [10]:
wlu = qu * strip_col * np.max([lx,ly]) / np.max([lx,ly])
Vo1 = wlu * np.max([lx,ly]) / 2
Vh1 = ((M_i_neg[0]*1e5) + (M_j_neg[0]*1e5))/np.max([lx,ly])

Vu  = (Vo1 + Vh1)

corte = SimpleShearSlab(fc = fc, fy = fy, bw = strip_col, H = H,rec = rec, fi = fiC,
                        diC = 1, Vu = Vu, lamda= 1, tc = tc, tl = tl, alfas = alfas)

corte.Vc_Slab()

..........................................................................................
Vu = 4.20 T
H = 20.00 cm
d = 17.50 cm
bw = 125.00 cm
bo = 230.00 cm
 λ = 1
 λs = 1.000
..........................................................................................
ACI 22.6.5.2
ΦVc (a) = 30.20 T
ΦVc (b) = 43.65 T
ΦVc (c) = 37.38 T
ΦVc (max) = 36.51 T
..........................................................................................
ΦVn = 30.20 T > Vu = 4.20 T -------> ok


### Flexural Desing

* Colum Strip

In [14]:
as_asmin_M_neg  = SimpleFlexuralCapacity(fc = fc, fy = fy, bw = strip_col, H = H, rec = rec,
                                         fi = fi, diF = diF, Mu = M_i_neg[0]*1e5)
[Asexc_csn, Asmin_csn, S_exc_csn, S_min_csn, numvar_exc_csn, numvar_min_csn, fiMn_csn, Mu_csn] = as_asmin_M_neg.as_asmin()

..........................................................................................
Mu = 5.18 T-m
H = 20.00 cm
d = 16.90 cm
..........................................................................................
Requeried extra Reinforcement
Asexc = 4.524 cm2
1 Φ 12 @ 31 / cant = 4 --> bw = 125 cm
Asmin = 4.524 cm2
1 Φ 12 @ 31 / cant = 4 --> bw = 125 cm
ΦMn = 5.67 T-m , Mu = 5.18 T-m, check = ok, DCR = 91 %
